# Phase 2 · Notebook 02 — Microsoft Presidio Baseline

[Microsoft Presidio](https://github.com/microsoft/presidio) is the de-facto industry tool for PII detection and redaction. It's a hybrid: a spaCy NER model under the hood plus a library of regex/checksum recognisers for things like phone numbers, IBANs, credit cards.

Two reasons it's worth testing here:

1. It's the most likely incumbent if a law firm has already done some PII work — measuring against it tells the buyer "you'd be better off than what you have today".
2. Its **regex recogniser layer** is exactly the right tool for TAB's `CODE` blind spot. We add a custom recogniser for ECHR application numbers (`Application no. 12345/67`) and check whether that closes the recall gap.

---


In [1]:
# ── Run me first if you're on Colab (skip locally — already in requirements.txt) ──
# !pip -q install transformers datasets evaluate seqeval accelerate \
#                 presidio-analyzer presidio-anonymizer scikit-learn spacy
# !python -m spacy download en_core_web_lg
# !python -m spacy download en_core_web_sm


## Setup


In [2]:
import sys
sys.path.insert(0, "../src")

import time
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

from anonymisation.data import load_tab
from anonymisation.evaluation import (
    evaluate_document, merge_results, results_to_dataframe,
)
from anonymisation.predictors import (
    build_presidio_analyzer, make_presidio_predictor, DEFAULT_PRESIDIO_LABEL_TO_TAB,
)


## Configuration


In [3]:
USE_FULL_TEST_SET = True
SAMPLE_SIZE = 100
MATCH_MODES = ["partial", "exact"]
PRESIDIO_THRESHOLD = 0.4         # drop predictions below this confidence
SPACY_BACKEND = "en_core_web_lg" # Presidio recommends `lg` for English

# We run TWO variants:
#   • baseline: stock Presidio recognisers
#   • +CODE:    same, but with our ECHR application-number recogniser added
RESULTS_PATH = "../results/baseline_presidio.csv"


## Build both analyzer variants


In [4]:
print(f"Building Presidio analyzers (spaCy backend: {SPACY_BACKEND})...")
analyzer_stock = build_presidio_analyzer(add_case_number_recognizer=False, spacy_model=SPACY_BACKEND)
analyzer_plus  = build_presidio_analyzer(add_case_number_recognizer=True,  spacy_model=SPACY_BACKEND)

predict_stock = make_presidio_predictor(analyzer_stock, threshold=PRESIDIO_THRESHOLD)
predict_plus  = make_presidio_predictor(analyzer_plus,  threshold=PRESIDIO_THRESHOLD)

# Smoke test on a synthetic ECHR-style sentence
demo = "Maria Petrova (Application no. 12345/67) is a Bulgarian national living in Plovdiv."
print("Stock recognisers:")
for s, e, t, txt in predict_stock(demo):
    print(f"  [{t:8s}] {txt!r}  ({s}:{e})")
print("\nWith CASE_NUMBER recogniser:")
for s, e, t, txt in predict_plus(demo):
    print(f"  [{t:8s}] {txt!r}  ({s}:{e})")


Building Presidio analyzers (spaCy backend: en_core_web_lg)...
Stock recognisers:
  [PERSON  ] 'Maria Petrova'  (0:13)
  [DEM     ] 'Bulgarian'  (46:55)
  [LOC     ] 'Plovdiv'  (75:82)

With CASE_NUMBER recogniser:
  [PERSON  ] 'Maria Petrova'  (0:13)
  [CODE    ] 'Application no. 12345/67'  (15:39)
  [DEM     ] 'Bulgarian'  (46:55)
  [LOC     ] 'Plovdiv'  (75:82)


## Load TAB & run the evaluation

Two passes — one per analyzer variant — so we can measure exactly how much the custom recogniser contributes.


In [5]:
dataset = load_tab()
test_docs = list(dataset["test"])
if not USE_FULL_TEST_SET:
    test_docs = test_docs[:SAMPLE_SIZE]
print(f"Evaluating on {len(test_docs)} TAB test documents.")

# Presidio is slower than spaCy/HF (it runs spaCy + every recogniser per doc)
# so be patient on the first pass.


Evaluating on 555 TAB test documents.


In [6]:
def run_eval(predict_fn, label):
    print(f"\n══ {label} ══")
    merged_by_mode = {}
    for mode in MATCH_MODES:
        print(f" -- {mode} match")
        per_doc = []
        start = time.time()
        for i, doc in enumerate(test_docs):
            if (i + 1) % 50 == 0:
                elapsed = time.time() - start
                print(f"    {i + 1}/{len(test_docs)}  ({(i + 1)/elapsed:.1f} docs/s)")
            per_doc.append(evaluate_document(predict_fn, doc, mode=mode))
        merged_by_mode[mode] = merge_results(per_doc)
        print(f"    done in {time.time() - start:.1f}s")
    return merged_by_mode

merged_stock = run_eval(predict_stock, "Presidio (stock)")
merged_plus  = run_eval(predict_plus,  "Presidio (+CASE_NUMBER recogniser)")



══ Presidio (stock) ══
 -- partial match
    50/555  (7.6 docs/s)
    100/555  (7.4 docs/s)
    150/555  (7.3 docs/s)
    200/555  (7.5 docs/s)
    250/555  (8.0 docs/s)
    300/555  (8.5 docs/s)
    350/555  (8.9 docs/s)
    400/555  (9.5 docs/s)
    450/555  (10.0 docs/s)
    500/555  (9.4 docs/s)
    550/555  (9.2 docs/s)
    done in 60.9s
 -- exact match
    50/555  (7.7 docs/s)
    100/555  (7.6 docs/s)
    150/555  (7.4 docs/s)
    200/555  (7.6 docs/s)
    250/555  (8.1 docs/s)
    300/555  (8.6 docs/s)
    350/555  (8.9 docs/s)
    400/555  (9.5 docs/s)
    450/555  (10.0 docs/s)
    500/555  (9.4 docs/s)
    550/555  (9.2 docs/s)
    done in 60.6s

══ Presidio (+CASE_NUMBER recogniser) ══
 -- partial match
    50/555  (7.7 docs/s)
    100/555  (7.6 docs/s)
    150/555  (7.4 docs/s)
    200/555  (7.5 docs/s)
    250/555  (8.0 docs/s)
    300/555  (8.5 docs/s)
    350/555  (8.9 docs/s)
    400/555  (9.5 docs/s)
    450/555  (10.0 docs/s)
    500/555  (9.4 docs/s)
    550/555  (

## Save results & compare


In [7]:
df_stock = results_to_dataframe(merged_stock)
df_stock.insert(0, "model", "presidio_stock")
df_plus = results_to_dataframe(merged_plus)
df_plus.insert(0, "model", "presidio_plus_case_number")
combined = pd.concat([df_stock, df_plus], ignore_index=True)
combined.to_csv(RESULTS_PATH, index=False)
print(f"Saved → {RESULTS_PATH}")

# Side-by-side overall and CODE rows
def overall_row(merged, mode, name):
    r = merged[mode]["_ALL"]
    return {"variant": name, "mode": mode, "P": f"{r.precision:.1%}", "R": f"{r.recall:.1%}", "F1": f"{r.f1:.1%}"}
def code_row(merged, mode, name):
    r = merged[mode]["CODE"]
    return {"variant": name, "mode": mode, "P": f"{r.precision:.1%}", "R": f"{r.recall:.1%}", "F1": f"{r.f1:.1%}"}

print("\nOverall (DIRECT + QUASI):")
print(pd.DataFrame([
    overall_row(merged_stock, "partial", "stock"),
    overall_row(merged_plus,  "partial", "+CASE_NUMBER"),
]).to_string(index=False))

print("\nCODE entity only — does the custom recogniser help?")
print(pd.DataFrame([
    code_row(merged_stock, "partial", "stock"),
    code_row(merged_plus,  "partial", "+CASE_NUMBER"),
]).to_string(index=False))


Saved → results/presidio_results.csv

Overall (DIRECT + QUASI):
     variant    mode     P     R    F1
       stock partial 49.3% 77.4% 60.2%
+CASE_NUMBER partial 49.4% 77.5% 60.3%

CODE entity only — does the custom recogniser help?
     variant    mode     P    R   F1
       stock partial  0.0% 0.0% 0.0%
+CASE_NUMBER partial 84.4% 1.7% 3.3%


## What to look for

- The stock Presidio numbers should be in the same ballpark as Phase 1's spaCy — under the hood it *is* spaCy, plus a few extra recognisers (phone numbers, IBANs) that don't apply to TAB.
- The custom `CASE_NUMBER` recogniser should take CODE recall from **0%** to something close to **100%**, because the ECHR application-number format is regex-friendly. That's the whole pitch for the regex layer in a real pipeline.
- The mosaic-effect implication is unchanged — even if regex closes CODE, the residual QUASI-fingerprint risk from Notebook 03 of Phase 1 still applies.

We'll bring these numbers into the head-to-head comparison in `04_head_to_head.ipynb`.
